# 01. Exploratory Data Analysis (EDA)
Performs initial inspection and visualization of the Massive Housing dataset.
Uses `PathConfig` for dynamic Colab/Windows path resolution.

In [ ]:
import sys, os

# --- Colab / Windows path bootstrap ---
if 'google.colab' in sys.modules:
    %cd /content/house-pridiction
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PathConfig, ModelConfig

print(f"Project root : {os.getcwd()}")
print(f"Raw data path: {PathConfig.RAW_DATA}")

## 1. Load Raw Dataset

In [ ]:
if not os.path.exists(PathConfig.RAW_DATA):
    print(f"❌ Raw data not found at: {PathConfig.RAW_DATA}")
    print("   Run: python -m src.data.make_dataset")
else:
    df = pd.read_csv(PathConfig.RAW_DATA)
    print(f"✅ Loaded raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
    df.head()

## 2. Basic Statistics

In [ ]:
print("=== Data Types ===")
print(df.dtypes.value_counts())
print("\n=== Missing Values (Top 10) ===")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(10))
print("\n=== Descriptive Statistics ===")
df.describe().T.round(2)

## 3. Target Variable: SalePrice
> **Note:** `SalePrice` in this dataset is already log-transformed. `np.expm1(value)` converts it back to dollars.

In [ ]:
target = ModelConfig.TARGET_COL
if target in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Log-scale distribution (as stored in CSV)
    axes[0].hist(df[target], bins=50, color='steelblue', edgecolor='white')
    axes[0].set_title('SalePrice Distribution (Log Scale - stored values)')
    axes[0].set_xlabel('log(SalePrice)')

    # Dollar-scale distribution (after inverse transform)
    axes[1].hist(np.expm1(df[target]), bins=50, color='darkorange', edgecolor='white')
    axes[1].set_title('SalePrice Distribution (Dollar Scale)')
    axes[1].set_xlabel('SalePrice ($)')

    plt.tight_layout()
    os.makedirs(os.path.join(PathConfig.REPORTS_DIR, 'figures'), exist_ok=True)
    plt.savefig(os.path.join(PathConfig.REPORTS_DIR, 'figures/sale_price_dist.png'), dpi=150)
    plt.show()
    print(f"\nMean price (dollars): ${np.expm1(df[target]).mean():,.0f}")
else:
    print(f"❌ Column '{target}' not found. Check your raw dataset.")

## 4. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include='number')
top_corr_features = numeric_df.corr()[target].abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(12, 8))
sns.heatmap(
    numeric_df[top_corr_features].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5
)
plt.title('Top 15 Feature Correlations with SalePrice')
plt.tight_layout()
plt.savefig(os.path.join(PathConfig.REPORTS_DIR, 'figures/correlation_heatmap.png'), dpi=150)
plt.show()